# Flux — scFEA metabolni flux (TRIM-Flux Var 2)

Izracuna metabolni flux za vsako celico prek **scFEA** (single-cell Flux Estimation Analysis).
Flux postane 3. modaliteta v TRIM (poleg RNA + TCR).

**Vhod:** `data_rna_counts.pkl` (SUROVI counti — scFEA jih sam normalizira; NE normalizirani `data_rna.pkl`)
**Izhod:** `data_flux.pkl` — matrika (celice x ~168 metabolnih modulov), poravnana z `data_labels`

Orodje scFEA (izbrano po raziskavi izvedljivosti): GNN, GPU, per-cell, dropout-robusten (korelacija >0.85),
~168 cloveskih metabolnih modulov. Nevzdrzevan od 2021 -> potrebni patchi za moderni Colab (spodaj).

> scFEA fluksi so RELATIVNI/model-odvisni (ne absolutne hitrosti); benchmark scFEA/Compass/METAFlux ne obstaja.


## 0. Mount + namestitev scFEA (+ patchi za moderni Colab)

In [ ]:
!pip install -q --upgrade numpy scanpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!git clone -q https://github.com/changwn/scFEA.git
%cd /content/scFEA

# scFEA importa 'magic' na vrhu skripte -> nujno nalozen (tudi pri sc_imputation=False).
# --no-deps: sicer vlece star pandas iz vira -> build pade.
!pip install -q --no-deps magic-impute graphtools scprep s_gd2 pygsp Deprecated tasklogger wrapt

# ============================================================================
# GLAVNI PATCH: vektoriziraj scFEA "process data" zanko.
# Original gradi geneExprDf z 168x rastocim concat -> O(n^2), ~80 MIN/batch
# (in RAM crash pri 70k). Vektorizirano = SEKUNDE, MATEMATICNO IDENTICNO
# (lokalno testirano: X + module_scale se ujemata, tudi prazni moduli).
# Patch skripto zapisemo z navadnim Pythonom (ne %%writefile, ki mora biti 1. vrstica).
# ============================================================================
patch_src = r'''
import sys
path = sys.argv[1]
src = open(path, encoding="utf-8").read()
start_marker = "geneExprDf = pd.DataFrame(columns"
end_marker   = "module_scale = torch.FloatTensor(module_scale.values/ moduleLen)"
i0 = src.find(start_marker); i1 = src.find(end_marker)
assert i0 != -1 and i1 != -1, "markerja ni najdena (scFEA ze patchan/drugacen?)"
i1 += len(end_marker)
line_start = src.rfind(chr(10), 0, i0) + 1
indent = src[line_start:i0]
vec = "# --- VEKTORIZIRANO (nadomesti 168x concat zanko; matematicno identicno) ---\n"
vec += indent + "emptyNode = []\n"
vec += indent + "gidx = {g: j for j, g in enumerate(gene_names)}\n"
vec += indent + "active = []\n"
vec += indent + "mask_rows = []\n"
vec += indent + "for i in range(n_modules):\n"
vec += indent + "    genes = [g for g in moduleGene.iloc[i, :].values.astype(str) if g != 'nan']\n"
vec += indent + "    if not genes:\n"
vec += indent + "        emptyNode.append(i)\n"
vec += indent + "        continue\n"
vec += indent + "    active.append(i)\n"
vec += indent + "    row = np.zeros(n_genes, dtype=bool)\n"
vec += indent + "    for g in genes:\n"
vec += indent + "        if g in gidx:\n"
vec += indent + "            row[gidx[g]] = True\n"
vec += indent + "    mask_rows.append(row)\n"
vec += indent + "mask = np.asarray(mask_rows)\n"
vec += indent + "G = geneExpr.values.astype('float32')\n"
vec += indent + "blocks = G[None, :, :] * mask[:, None, :]\n"
vec += indent + "X = blocks.transpose(1, 0, 2).reshape(n_cells, len(active) * n_genes).astype('float32')\n"
vec += indent + "X = torch.FloatTensor(X).to(device)\n"
vec += indent + "mlen_active = np.asarray(moduleLen)[active].astype('float64')\n"
vec += indent + "module_scale = (blocks.sum(axis=2).T.astype('float64') / mlen_active)\n"
vec += indent + "module_scale = torch.FloatTensor(module_scale)"
open(path, "w", encoding="utf-8").write(src[:line_start] + vec + src[i1:])
import ast; ast.parse(open(path, encoding="utf-8").read())
print("process-data VEKTORIZIRAN + sintaksa OK")
'''
with open('/content/patch_processdata.py', 'w') as f:
    f.write(patch_src)
print('patch skripta zapisana -> /content/patch_processdata.py')

In [ ]:
# Apliciraj patche na scFEA.py (vrstni red pomemben!)
# 1) GLAVNI: vektoriziraj process-data (resi 80min/batch + RAM crash)
!python /content/patch_processdata.py /content/scFEA/src/scFEA.py

# 2) torch: inference (vrstici ~299-300) da .detach().numpy() na GPU tensorju -> pade.
#    Dodaj .cpu(). (Ta vrstica NI del vektoriziranega bloka -> se vedno potrebna.)
!sed -i 's/\.detach()\.numpy()/.detach().cpu().numpy()/g' src/scFEA.py

# 3) (varovalno, neskodljivo) stari pandas 3 sed patchi -- ce vektorizacija ne bi
#    prijela, ti se vedno popravijo append/dtype. Po vektorizaciji ne najdejo nic.
!sed -i 's/geneExprDf = geneExprDf\.append(temp,/geneExprDf = pd.concat([geneExprDf, temp],/' src/scFEA.py
!sed -i 's/X = geneExprDf\.values\.T/X = geneExprDf.values.T.astype("float32")/' src/scFEA.py

# preveri, da je scFEA se vedno veljaven python + magic uvozljiv
!python -c "import ast; ast.parse(open('src/scFEA.py').read()); print('scFEA.py sintaksa OK')"
import magic
print('magic:', getattr(magic, '__version__', 'OK'))
print('cmMat na voljo:')
!ls data/ | grep -iE 'module_gene|cmMat'

## 1. Nalozi surove counte + gene imena

scFEA hoce **surove counte** (sam logira ce max>50) in **gene simbole** (vrstice=geni, stolpci=celice).
`data_rna_counts.pkl` je shranjen v notebooku 01 (loceno od normaliziranega `data_rna.pkl`).

In [ ]:
import pickle, numpy as np, pandas as pd, os, time
from scipy.sparse import issparse

DATA = '/content/drive/MyDrive/Diploma/data/processed'

# === MONTE CARLO FLUX SEED ===
# scFEA je nedeterministicen (GNN random init) -> vsak zagon da rahlo drugacen flux.
# Za Monte Carlo poZeni 3x s flux_seed 0,1,2 -> data_flux_seed{s}.pkl (sparovano s TRIM seed).
flux_seed = 0     # spremeni za ponovitve: 0, 1, 2
print(f'FLUX SEED: {flux_seed} -> izhod flux_runs/data_flux_seed{flux_seed}.pkl')

with open(os.path.join(DATA, 'data_rna_counts.pkl'), 'rb') as f:
    counts = pickle.load(f)                 # SUROVI counti (celice x geni), sparse
with open(os.path.join(DATA, 'gene_names.pkl'), 'rb') as f:
    gene_names = [str(g) for g in pickle.load(f)]

print('counts:', counts.shape, type(counts).__name__)
print('gene_names:', len(gene_names), '| primer:', gene_names[:4])
assert len(gene_names) == counts.shape[1], 'gene_names != stolpci counts!'

# scFEA rabi gene SIMBOLE (CD8A), ne Ensembl ID (ENSG...).
is_ensembl = all(g.upper().startswith('ENSG') for g in gene_names[:50])
assert not is_ensembl, 'Geni so Ensembl ID -> scFEA rabi simbole (pretvori z mygene)!'
print('Geni so simboli:', not is_ensembl)

In [ ]:
# === DIAGNOSTIKA GPU (pred dolgim zagonom!) ===
# Colab kaze le 0.4 GB GPU RAM -> preveri ce scFEA sploh uporablja GPU.
# scFEA (scFEA.py:94): device = cuda ce torch.cuda.is_available() SICER cpu (tiho!).
import torch
print('torch:', torch.__version__)
print('CUDA na voljo (torch.cuda.is_available()):', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    # hiter GPU test: majhen matmul + koliko RAM zasede
    x = torch.randn(5000, 658, device='cuda')
    w = torch.randn(658, 168, device='cuda')
    _ = (x @ w).sum().item()
    print(f'GPU test OK | zaseden GPU RAM: {torch.cuda.memory_allocated()/1e9:.3f} GB '
          f'(majhen model -> malo RAM je NORMALNO, ne dokaz da GPU ne dela)')
    print('-> scFEA BO uporabil GPU (is_available=True). 0.4 GB je OK za tako majhen GNN.')
    del x, w; torch.cuda.empty_cache()
else:
    print('!!! CUDA NI na voljo -> scFEA bo tekel na CPU (10x+ pocasneje = 2h/batch).')
    print('!!! USTAVI: Runtime > Change runtime type > Hardware accelerator > GPU (T4/A100).')
# preveri tudi Colab nvidia-smi
!nvidia-smi --query-gpu=name,memory.used,memory.total,utilization.gpu --format=csv 2>/dev/null || echo 'nvidia-smi ni na voljo -> verjetno CPU runtime'

## 2. Batch scFEA -> flux (RESUMABLE, za nocni zagon)

70k celic naenkrat crasha RAM (scFEA "process data" zanka = 168 kopij matrike),
zato kosi po CHUNK celic. scFEA trenira GNN 100 epoch/batch -> DOLGO (~ure/batch).

**RESUMABLE:** vsak batch se takoj shrani na **Drive** (ne lokalni disk!). Ob resetu
runtime (Colab pogosto odklopi cez noc) samo znova zazeni celico -> preskoci ze
narejene batche, nadaljuje kjer je ostal. NIC se ne izgubi.

Prvi batch IZPISE device (cuda vs cpu). Ce "cpu" -> trening bo 10x pocasnejsi ->
ustavi, preklopi na GPU runtime (Runtime > Change runtime type > GPU).

Izhod: `data_flux.pkl` (celice x ~168 modulov), poravnan z `data_labels`.

In [ ]:
# ============================================================================
# BATCH scFEA -> flux (RESUMABLE)
#
# Po vektorizaciji process-data (cell-3) je scFEA hiter: ~5 min/batch (trening).
# Kosi po CHUNK celic (RAM). Vsak flux CSV -> na DRIVE (prezivi reset).
# CE CRASHA: znova zazeni celico -> preskoci narejene -> nadaljuje. NIC rocno.
# ============================================================================
from tqdm import tqdm
import json

FLUX_DIR = os.path.join(DATA, 'flux_batches', f'seed{flux_seed}')   # per-seed: 3 zagoni se ne mesajo
os.makedirs(FLUX_DIR, exist_ok=True)
print('Batch izhodi (Drive, resumable):', FLUX_DIR)

# 1) filtriraj na scFEA modulne gene (55x manjsi vhod, identicen rezultat)
mg = pd.read_csv('/content/scFEA/data/module_gene_m168.csv', index_col=0)
module_genes = set()
for col in mg.columns:
    for v in mg[col].dropna().astype(str):
        g = v.strip()
        if g and g.lower() != 'nan': module_genes.add(g)
keep_idx = [i for i, g in enumerate(gene_names) if g in module_genes]
keep_names = [gene_names[i] for i in keep_idx]
print(f'scFEA modulnih genov: {len(module_genes)} | nasih v modulih: {len(keep_idx)}')
assert len(keep_idx) > 100, 'premalo ujemanja gene-imen!'

counts_k = counts[:, keep_idx]
N = counts_k.shape[0]
CHUNK = 5000
n_batches = (N + CHUNK - 1) // CHUNK

# VAROVALKA: CHUNK se ne sme spremeniti med resume (sicer c{i} indeksi razni)
meta_path = os.path.join(FLUX_DIR, 'run_meta.json')
if os.path.exists(meta_path):
    meta = json.load(open(meta_path))
    assert meta['N'] == N and meta['CHUNK'] == CHUNK, (
        f"NEUJEMANJE! Zacel z N={meta['N']}, CHUNK={meta['CHUNK']}, zdaj N={N}, CHUNK={CHUNK}. "
        f"Vrni CHUNK={meta['CHUNK']} ali izbrisi {FLUX_DIR}.")
else:
    json.dump({'N': int(N), 'CHUNK': int(CHUNK)}, open(meta_path, 'w'))
print(f'Celic: {N} | kosov po {CHUNK}: {n_batches}')

os.makedirs('/content/scfea_input', exist_ok=True)
%cd /content/scFEA

done = [b for b in range(n_batches) if os.path.exists(os.path.join(FLUX_DIR, f'flux_{b}.csv'))]
if done:
    print(f'RESUME: ze narejeni {done} ({len(done)}/{n_batches}) -> preskocim.')

t0 = time.time()
for b in tqdm(range(n_batches), desc='scFEA batchi'):
    out_csv = os.path.join(FLUX_DIR, f'flux_{b}.csv')
    if os.path.exists(out_csv):
        continue

    lo, hi = b*CHUNK, min((b+1)*CHUNK, N)
    Xk = counts_k[lo:hi]
    Xk = Xk.toarray() if issparse(Xk) else np.asarray(Xk)
    Xk = np.rint(Xk).astype(np.int32)
    df_in = pd.DataFrame(Xk.T, index=keep_names, columns=[f'c{i}' for i in range(lo, hi)])
    df_in.to_csv('/content/scfea_input/expr.csv')
    del Xk, df_in

    # scFEA -> flux + log LOKALNO (/content, hitro; ne Drive/FUSE)
    tmp_csv = f'/content/flux_{b}.csv'
    local_log = f'/content/log_{b}.txt'
    rc = os.system(
        f'python src/scFEA.py --data_dir data --input_dir /content/scfea_input '
        f'--test_file expr.csv --moduleGene_file module_gene_m168.csv '
        f'--stoichiometry_matrix cmMat_c70_m168.csv '
        f'--output_flux_file {tmp_csv} --output_balance_file /content/bal_{b}.csv '
        f'--sc_imputation False > "{local_log}" 2>&1')

    if rc != 0 or not os.path.exists(tmp_csv):
        print(f'\n!!! batch {b} PADEL -- zadnjih 20 vrstic log:')
        os.system(f'tail -n 20 "{local_log}"')
        os.system(f'cp "{local_log}" "{os.path.join(FLUX_DIR, f"log_{b}.txt")}"')  # na Drive za post-mortem
        raise RuntimeError(f'batch {b} padel')

    # preberi + preveri (shape + brez NaN) + shrani na Drive (atomarno)
    fb = pd.read_csv(tmp_csv, index_col=0)
    assert fb.shape[0] == (hi - lo), f'batch {b}: {fb.shape[0]} != {hi-lo}!'
    assert int(np.isnan(fb.values).sum()) == 0, f'batch {b} NaN!'
    fb.to_csv(out_csv)
    os.remove(tmp_csv)
    print(f'  batch {b}: {fb.shape} shranjen ({time.time()-t0:.0f}s skupaj)')

print(f'\n=== vsi batchi koncani: {time.time()-t0:.0f} s ===')

# 2) zdruzi VSE batch CSV-je z DRIVE (deluje tudi po resetu)
flux_parts = []
for b in range(n_batches):
    fp = os.path.join(FLUX_DIR, f'flux_{b}.csv')
    assert os.path.exists(fp), f'MANJKA batch {b} ({fp})! Znova zazeni celico.'
    flux_parts.append(pd.read_csv(fp, index_col=0))
flux = pd.concat(flux_parts, axis=0)
print('Zdruzen flux:', flux.shape)

# 3) poravnaj na NAS vrstni red celic (c0..cN-1) + preveri
expected = [f'c{i}' for i in range(N)]
assert set(expected) == set(flux.index.astype(str)), 'flux celice se ne ujemajo!'
flux = flux.loc[expected]
n_nan = int(np.isnan(flux.values).sum())
print('NaN:', n_nan, '| delez nicelnih:', f'{(flux.values==0).mean():.3f}',
      '| min/max/mean:', f'{flux.values.min():.3f}/{flux.values.max():.3f}/{flux.values.mean():.3f}')
assert n_nan == 0, 'FLUX VSEBUJE NaN!'
assert flux.shape[0] == counts.shape[0], f'flux vrstic != celic!'

# 4) shrani koncni flux -> flux_runs/data_flux_seed{s}.pkl (per-seed za Monte Carlo)
data_flux = flux.values.astype(np.float32)
FLUX_RUNS = os.path.join(DATA, 'flux_runs')
os.makedirs(FLUX_RUNS, exist_ok=True)
out_pkl = os.path.join(FLUX_RUNS, f'data_flux_seed{flux_seed}.pkl')
with open(out_pkl, 'wb') as f:
    pickle.dump(data_flux, f)
with open(os.path.join(FLUX_RUNS, 'flux_module_names.pkl'), 'wb') as f:
    pickle.dump(list(flux.columns), f)
print(f'\n{out_pkl} shranjen: {data_flux.shape} (flux_seed={flux_seed})')

## 5. Zakljucek

- `data_flux.pkl` (celice x ~168 modulov) = metabolni flux, poravnan z RNA/TCR.
- Naslednje: flux encoder/decoder kot 3. modaliteta v TRIM (RNA + TCR + Flux, Var 2).

Pridrzki za diplomo: scFEA fluksi RELATIVNI/model-odvisni; scFEA nevzdrzevan (patchi za pandas/torch).